# Module 7: Clinical NLP — Live Walkthrough
**MPHY 6120 · AI for Medicine**

Walk through alongside the lecture slides. Each section maps to a slide group.

---

In [ ]:
# Setup — run this first
# !uv pip install transformers scikit-learn matplotlib numpy
# !uv pip install scispacy
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz

import warnings
warnings.filterwarnings('ignore')

---
## 1. Clinical Text: What Makes It Hard?
*Slides 5-11*

Let's look at real (synthetic) clinical text and see why standard NLP breaks.

In [ ]:
# A synthetic radiology report — structured sections
radiology_report = """
INDICATION: 45-year-old female with shortness of breath.

TECHNIQUE: PA and lateral chest radiograph.

FINDINGS:
The lungs are clear without focal consolidation, pleural
effusion, or pneumothorax. Heart size is normal.
Mediastinal contours are unremarkable.

IMPRESSION:
No acute cardiopulmonary process.
"""

# A synthetic progress note — messy, abbreviated
progress_note = """
S: Pt c/o SOB x 2 days, worse w/ exertion. Denies CP,
   orthopnea. Hx of CHF, COPD. Stopped taking lasix 3d ago.

O: VS: T 98.6, HR 92, BP 142/88, RR 22, SpO2 92% RA
   Lungs: Bibasilar crackles, no wheezes
   Ext: 2+ pitting edema bilat LE

A/P: Acute on chronic CHF exacerbation, likely d/t med
     non-compliance. Restart lasix 40mg BID.
"""

print("=== RADIOLOGY REPORT ===")
print(radiology_report)
print("=== PROGRESS NOTE ===")
print(progress_note)

In [ ]:
# The abbreviation problem
# Same abbreviation, totally different meanings:

ambiguous = {
    "MS":  ["multiple sclerosis", "mental status", "morphine sulfate", "mitral stenosis"],
    "PT":  ["patient", "physical therapy", "prothrombin time"],
    "DC":  ["discontinue", "discharge", "direct current"],
    "SOB": ["shortness of breath"],  # but general English...
    "CP":  ["chest pain", "cerebral palsy", "cleft palate"],
}

for abbrev, meanings in ambiguous.items():
    print(f"  {abbrev:4s} → {' | '.join(meanings)}")

print("\n💡 Context is everything. No lookup table can solve this.")

In [ ]:
# Negation: why keyword matching fails

sentences = [
    "Patient has pneumonia",              # PRESENT
    "No evidence of pneumonia",           # ABSENT
    "Patient denies chest pain",          # ABSENT
    "Rule out PE",                        # UNCERTAIN
    "Unlikely malignancy",                # UNCERTAIN
    "Patient's mother has diabetes",      # FAMILY (not patient!)
    "History of stroke, now resolved",    # HISTORICAL
]

# Naive keyword search
print("Naive keyword search for 'pneumonia':")
for s in sentences:
    if "pneumonia" in s.lower():
        print(f"  ✓ FOUND: {s}")
    else:
        print(f"  ✗ miss:  {s}")

print("\n🚨 'No evidence of pneumonia' is NOT pneumonia!")
print("   Simple keyword matching would flag this as positive.")
print("   This is why we need NLP, not regex.")

---
## 2. Tokenization
*Slides 14-15*

Breaking text into pieces the model can understand.

In [ ]:
# Simple whitespace tokenization
text = "Patient has SOB and CP. BP 142/88mmHg."

# Method 1: split on whitespace
tokens_naive = text.split()
print("Whitespace split:")
print(f"  {tokens_naive}")
print(f"  Problem: 'CP.' has punctuation attached, '142/88mmHg.' is one token")

print()

In [ ]:
# Method 2: BERT tokenizer (subword)
from transformers import AutoTokenizer

# General-purpose BERT
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

text = "Patient has SOB and CP"
tokens = tokenizer.tokenize(text)
ids = tokenizer.encode(text)

print(f"Text:   '{text}'")
print(f"Tokens: {tokens}")
print(f"IDs:    {ids}")
print()
print("Notice: 'sob' gets split into 'so' + '##b'")
print("BERT doesn't know 'SOB' = shortness of breath!")

In [ ]:
# Method 3: Clinical BERT tokenizer
clinical_tokenizer = AutoTokenizer.from_pretrained(
    "emilyalsentzer/Bio_ClinicalBERT"
)

texts = [
    "Patient has SOB and CP",
    "Bibasilar crackles, no wheezes",
    "Acute on chronic CHF exacerbation",
    "non-small-cell lung carcinoma",
    "BP 142/88mmHg SpO2 92%",
]

print(f"{'Text':<45} {'General BERT':<35} {'ClinicalBERT'}")
print("-" * 115)
for t in texts:
    gen = tokenizer.tokenize(t)
    clin = clinical_tokenizer.tokenize(t)
    print(f"{t:<45} {str(gen):<35} {clin}")

In [ ]:
# Vocabulary comparison
print(f"General BERT vocab size:  {tokenizer.vocab_size:,}")
print(f"ClinicalBERT vocab size:  {clinical_tokenizer.vocab_size:,}")
print()

# Check if medical terms are in vocabulary
medical_terms = ["pneumonia", "cardiomegaly", "atelectasis", "effusion",
                 "hemoptysis", "dyspnea", "tachycardia", "SOB"]

print(f"{'Term':<20} {'In general BERT?':<20} {'In ClinicalBERT?'}")
print("-" * 56)
for term in medical_terms:
    gen_tokens = tokenizer.tokenize(term)
    clin_tokens = clinical_tokenizer.tokenize(term)
    gen_whole = len(gen_tokens) == 1
    clin_whole = len(clin_tokens) == 1
    print(f"{term:<20} {'✓ whole' if gen_whole else '✗ ' + str(gen_tokens):<20} "
          f"{'✓ whole' if clin_whole else '✗ ' + str(clin_tokens)}")

---
## 3. Text Normalization
*Slide 16*

Cleaning text before feeding it to models.

In [ ]:
import re

# Common clinical abbreviation dictionary
ABBREV_MAP = {
    "pt": "patient", "pts": "patients",
    "hx": "history", "dx": "diagnosis", "tx": "treatment", "rx": "prescription",
    "sob": "shortness of breath", "cp": "chest pain",
    "chf": "congestive heart failure", "copd": "chronic obstructive pulmonary disease",
    "c/o": "complains of", "w/": "with", "d/t": "due to",
    "bid": "twice daily", "tid": "three times daily", "prn": "as needed",
    "bilat": "bilateral", "le": "lower extremity",
}

def normalize_clinical_text(text):
    """Basic clinical text normalization."""
    # Lowercase
    text = text.lower()
    # Remove common artifacts
    text = re.sub(r'\*{3}.*?\*{3}', '', text)   # *** DRAFT ***
    text = re.sub(r'\[.*?\]', '', text)          # [timestamp]
    # Expand abbreviations (word boundary matching)
    for abbrev, expansion in ABBREV_MAP.items():
        text = re.sub(r'\b' + re.escape(abbrev) + r'\b', expansion, text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

raw = "Pt c/o SOB x 2 days, worse w/ exertion. Hx of CHF, COPD."
normalized = normalize_clinical_text(raw)

print(f"Raw:        {raw}")
print(f"Normalized: {normalized}")

In [ ]:
# Section segmentation — extract structured parts from a report

def extract_sections(report):
    """Split a radiology report into sections."""
    sections = {}
    current_section = "PREAMBLE"
    current_text = []
    
    for line in report.strip().split('\n'):
        line = line.strip()
        # Check if line is a section header (all caps followed by colon)
        if re.match(r'^[A-Z][A-Z /]+:', line):
            if current_text:
                sections[current_section] = ' '.join(current_text).strip()
            current_section = line.split(':')[0].strip()
            remainder = ':'.join(line.split(':')[1:]).strip()
            current_text = [remainder] if remainder else []
        else:
            current_text.append(line)
    
    if current_text:
        sections[current_section] = ' '.join(current_text).strip()
    
    return sections

sections = extract_sections(radiology_report)
for section, text in sections.items():
    print(f"[{section}]")
    print(f"  {text}")
    print()

print("💡 For classification, IMPRESSION alone is often enough.")
print(f"   Full report: {len(radiology_report.split())} words")
print(f"   Impression:  {len(sections.get('IMPRESSION', '').split())} words")

---
## 4. Word Embeddings
*Slides 17-18*

From words to numbers — how models represent meaning.

In [ ]:
# Bag of Words — the simplest representation
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

corpus = [
    "No acute cardiopulmonary process",
    "Large right pleural effusion with atelectasis",
    "Lungs are clear bilaterally",
    "Bilateral pleural effusions, moderate",
    "Normal cardiac silhouette, clear lungs",
    "Patchy opacity right lower lobe, possible pneumonia",
]

labels = ["Normal", "Abnormal", "Normal", "Abnormal", "Normal", "Abnormal"]

# Count vectorizer
count_vec = CountVectorizer()
X_count = count_vec.fit_transform(corpus)

print("Bag of Words (first 2 documents):")
vocab = count_vec.get_feature_names_out()
for i in range(2):
    nonzero = X_count[i].nonzero()[1]
    words = [(vocab[j], X_count[i, j]) for j in nonzero]
    print(f"  '{corpus[i]}'")
    print(f"   → {dict(words)}")
    print()

In [ ]:
# TF-IDF — words that appear everywhere are less informative
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

# What are the most discriminative words?
import numpy as np

feature_names = tfidf.get_feature_names_out()
mean_tfidf = np.array(X_tfidf.mean(axis=0)).flatten()
top_idx = mean_tfidf.argsort()[::-1][:10]

print("Top TF-IDF terms across corpus:")
for idx in top_idx:
    print(f"  {feature_names[idx]:<20} {mean_tfidf[idx]:.3f}")

print("\n💡 'effusion' and 'pleural' are highly weighted — they're specific.")
print("   'are' and 'with' get low weight — they appear everywhere.")

In [ ]:
# Quick classifier: TF-IDF + Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

y = np.array([0, 1, 0, 1, 0, 1])  # 0=Normal, 1=Abnormal

clf = LogisticRegression()
clf.fit(X_tfidf, y)

# What did it learn?
coefs = clf.coef_[0]
top_abnormal = np.argsort(coefs)[-5:]
top_normal = np.argsort(coefs)[:5]

print("Words predicting ABNORMAL:")
for idx in reversed(top_abnormal):
    print(f"  +{coefs[idx]:+.2f}  {feature_names[idx]}")

print("\nWords predicting NORMAL:")
for idx in top_normal:
    print(f"  {coefs[idx]:+.2f}  {feature_names[idx]}")

# Test on new reports
new_reports = [
    "Small left pleural effusion",
    "Clear lungs, no acute findings",
    "Right lower lobe consolidation concerning for pneumonia",
]

X_new = tfidf.transform(new_reports)
preds = clf.predict(X_new)
probs = clf.predict_proba(X_new)

print("\nPredictions on new reports:")
for report, pred, prob in zip(new_reports, preds, probs):
    label = "Abnormal" if pred == 1 else "Normal"
    print(f"  {label} ({prob[1]:.0%}): {report}")

In [ ]:
# BERT Embeddings — contextual representations
from transformers import AutoModel
import torch

model_name = "emilyalsentzer/Bio_ClinicalBERT"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

def get_embedding(text):
    """Get ClinicalBERT [CLS] embedding for a text."""
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # [CLS] token embedding = sentence representation
    return outputs.last_hidden_state[0, 0].numpy()

# Compare embeddings of similar/different sentences
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

pairs = [
    ("chest pain", "cardiac discomfort"),         # Similar meaning
    ("chest pain", "knee pain"),                   # Same structure, different
    ("no pneumonia", "pneumonia present"),          # Opposite meaning
    ("shortness of breath", "SOB"),                # Abbreviation
    ("normal cardiac silhouette", "cardiomegaly"), # Opposite findings
]

print(f"{'Text A':<30} {'Text B':<30} {'Cosine Sim':>10}")
print("-" * 72)
for a, b in pairs:
    emb_a = get_embedding(a)
    emb_b = get_embedding(b)
    sim = cosine_sim(emb_a, emb_b)
    print(f"{a:<30} {b:<30} {sim:>10.3f}")

print("\n💡 ClinicalBERT captures medical meaning in context.")
print("   'chest pain' ≈ 'cardiac discomfort' (high similarity)")
print("   'no pneumonia' ≠ 'pneumonia present' (lower similarity)")

---
## 5. Named Entity Recognition (NER)
*Slides 19-25*

Extracting structured information from unstructured text.

In [ ]:
# NER with scispaCy (if installed)
try:
    import spacy
    nlp = spacy.load("en_core_sci_md")
    
    text = ("Patient presents with chest pain and shortness of breath. "
            "History of type 2 diabetes mellitus and hypertension. "
            "Currently taking metformin 500mg and lisinopril 10mg.")
    
    doc = nlp(text)
    
    print("Entities found by scispaCy:")
    print(f"{'Entity':<35} {'Label':<15} {'Start':<6} {'End'}")
    print("-" * 65)
    for ent in doc.ents:
        print(f"{ent.text:<35} {ent.label_:<15} {ent.start_char:<6} {ent.end_char}")
    
    print(f"\nTotal entities: {len(doc.ents)}")

except (ImportError, OSError) as e:
    print("scispaCy not installed — showing expected output instead:")
    print()
    expected = [
        ("chest pain",                "ENTITY"),
        ("shortness of breath",       "ENTITY"),
        ("type 2 diabetes mellitus",  "ENTITY"),
        ("hypertension",              "ENTITY"),
        ("metformin",                 "ENTITY"),
        ("lisinopril",                "ENTITY"),
    ]
    print(f"{'Entity':<35} {'Label'}")
    print("-" * 50)
    for ent, label in expected:
        print(f"{ent:<35} {label}")
    print("\n(Install scispaCy to see live results)")

In [ ]:
# Simple rule-based negation detection (NegEx-style)

NEGATION_CUES = [
    "no", "not", "denies", "denied", "without", "no evidence of",
    "negative for", "rules out", "ruled out", "absent",
    "unremarkable", "free of", "clear of"
]

UNCERTAINTY_CUES = [
    "rule out", "possible", "probable", "likely", "unlikely",
    "cannot exclude", "suspicious for", "concerning for", "question of"
]

def detect_assertion(sentence, entity):
    """Simple assertion detection for an entity in a sentence."""
    s = sentence.lower()
    e = entity.lower()
    
    if e not in s:
        return "NOT_FOUND"
    
    # Check window before entity
    idx = s.index(e)
    window = s[:idx]
    
    for cue in NEGATION_CUES:
        if cue in window:
            return "ABSENT"
    
    for cue in UNCERTAINTY_CUES:
        if cue in window:
            return "UNCERTAIN"
    
    return "PRESENT"

test_cases = [
    ("Patient has pneumonia", "pneumonia"),
    ("No evidence of pneumonia", "pneumonia"),
    ("Patient denies chest pain", "chest pain"),
    ("Rule out pulmonary embolism", "pulmonary embolism"),
    ("Possible pneumonia in right lower lobe", "pneumonia"),
    ("Lungs clear without effusion", "effusion"),
    ("History of stroke, now resolved", "stroke"),
]

print(f"{'Sentence':<50} {'Entity':<25} {'Assertion'}")
print("-" * 90)
for sentence, entity in test_cases:
    assertion = detect_assertion(sentence, entity)
    marker = {"PRESENT": "🔴", "ABSENT": "🟢", "UNCERTAIN": "🟡"}.get(assertion, "⚪")
    print(f"{sentence:<50} {entity:<25} {marker} {assertion}")

print("\n💡 This is a toy version of NegEx. Real systems use dependency")
print("   parsing and scope detection. But even this catches most cases.")

---
## 6. Putting It Together: Report Classification
*Slides 26-33*

Building a classifier to distinguish normal vs abnormal radiology reports.

In [ ]:
# Larger synthetic dataset of radiology impressions
impressions = [
    # Normal
    ("No acute cardiopulmonary process.", 0),
    ("Clear lungs bilaterally. Normal cardiac silhouette.", 0),
    ("No focal consolidation, effusion, or pneumothorax.", 0),
    ("Unremarkable chest radiograph.", 0),
    ("No acute abnormality.", 0),
    ("Lungs are clear. Heart size is normal.", 0),
    ("No evidence of acute disease.", 0),
    ("Normal examination.", 0),
    ("No significant interval change from prior.", 0),
    ("Stable appearance of the chest.", 0),
    # Abnormal
    ("Right lower lobe consolidation consistent with pneumonia.", 1),
    ("Large right pleural effusion with adjacent atelectasis.", 1),
    ("Bilateral pleural effusions, moderate.", 1),
    ("Patchy opacity right lower lobe, possible pneumonia.", 1),
    ("Cardiomegaly with pulmonary vascular congestion.", 1),
    ("Left apical pneumothorax.", 1),
    ("New right-sided rib fractures.", 1),
    ("Widened mediastinum concerning for aortic pathology.", 1),
    ("Interstitial prominence suggesting pulmonary edema.", 1),
    ("Mass-like opacity right hilum, recommend CT.", 1),
]

texts = [t for t, _ in impressions]
labels = np.array([l for _, l in impressions])

print(f"Dataset: {len(texts)} impressions ({(labels==0).sum()} normal, {(labels==1).sum()} abnormal)")

In [ ]:
# Approach 1: TF-IDF + Logistic Regression (classic ML)
from sklearn.pipeline import Pipeline
from sklearn.model_selection import LeaveOneOut, cross_val_predict

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=100)),
    ('clf', LogisticRegression(max_iter=1000))
])

# Leave-one-out cross-validation (small dataset)
loo = LeaveOneOut()
preds = cross_val_predict(pipeline, texts, labels, cv=loo)
accuracy = (preds == labels).mean()

print(f"TF-IDF + LogReg (LOO-CV): {accuracy:.0%} accuracy")
print()

# Show errors
errors = np.where(preds != labels)[0]
if len(errors) > 0:
    print("Misclassified:")
    for idx in errors:
        true = "Abnormal" if labels[idx] == 1 else "Normal"
        pred = "Abnormal" if preds[idx] == 1 else "Normal"
        print(f"  True={true}, Pred={pred}: '{texts[idx]}'")
else:
    print("Perfect classification! (small dataset though)")

print("\n💡 TF-IDF works surprisingly well for this task because")
print("   radiology impressions use very specific vocabulary.")
print("   'clear' → normal, 'effusion/consolidation' → abnormal.")

In [ ]:
# Approach 2: ClinicalBERT embeddings + Logistic Regression
print("Computing ClinicalBERT embeddings...")

embeddings = np.array([get_embedding(text) for text in texts])
print(f"Embedding shape: {embeddings.shape}")

from sklearn.linear_model import LogisticRegression

preds_bert = cross_val_predict(
    LogisticRegression(max_iter=1000),
    embeddings, labels, cv=loo
)
accuracy_bert = (preds_bert == labels).mean()

print(f"\nClinicalBERT + LogReg (LOO-CV): {accuracy_bert:.0%} accuracy")
print(f"TF-IDF + LogReg (LOO-CV):       {accuracy:.0%} accuracy")

print("\n💡 On a tiny dataset like this, TF-IDF often matches BERT.")
print("   BERT shines on larger datasets and harder tasks where")
print("   contextual understanding matters (negation, ambiguity).")

---
## Key Takeaways

1. **Clinical text is uniquely challenging** — abbreviations, negation, implicit context, PHI
2. **Tokenization matters** — ClinicalBERT handles medical terms better than general BERT
3. **Simple approaches work** — TF-IDF + LogReg is a strong baseline for many clinical NLP tasks
4. **Negation detection is critical** — "no pneumonia" ≠ "pneumonia"
5. **Context is everything** — BERT embeddings capture meaning, not just word frequency

### Next: Module 8 — LLMs in Medicine
From BERT (understanding) → GPT (generation) → Clinical applications